 # Week 5 - Data Cleaning using PySpark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder \
    .appName("Week5_Data_Cleaning") \
    .getOrCreate()

In [3]:
df = spark.read.csv(
    "../data/sales_data.csv",
    header=True,
    inferSchema=True
)

In [4]:
df.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-----+--------+-------+--------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  status|price|store_id|revenue|         email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-----+--------+-------+--------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|    Delhi| 25|     Premium|    NULL|  100|       1|    500|john@gmail.com|    john|2025-01-01 10:00:00|
|    101|      2025-01-01|  West|     Electronics|        500|    Delhi| 25|     Premium|    NULL|  100|       1|    500|john@gmail.com|    john|2025-01-01 10:00:00|
|    102|      2025-01-02|  East|        Clothing|        300|   Mumbai| 19|       Basic|  Active| NULL|       2|    300| sam@gmail.com|     sam|2025-01-02 11:00:00|
|   

## Q1. Limitations of MapReduce

1. Heavy disk I/O
2. Slow iterative processing
3. Complex programming model
4. No real-time processing
5. Limited analytics support

Spark solves these issues using in-memory computing and unified APIs.

## Q2. In-Memory Computing

Spark stores intermediate data in RAM instead of repeatedly writing to disk. This makes iterative machine learning algorithms significantly faster than MapReduce.

## Q3. Remove Duplicates

In [5]:
df_q3 = df.dropDuplicates(
    ["user_id","transaction_date"]
)

df_q3.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-----+--------+-------+--------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  status|price|store_id|revenue|         email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-----+--------+-------+--------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|    Delhi| 25|     Premium|    NULL|  100|       1|    500|john@gmail.com|    john|2025-01-01 10:00:00|
|    102|      2025-01-02|  East|        Clothing|        300|   Mumbai| 19|       Basic|  Active| NULL|       2|    300| sam@gmail.com|     sam|2025-01-02 11:00:00|
|    103|      2025-01-03|  West|       Furniture|        700|    Delhi| 28|     Premium|    NULL|  200|       1|    700|          NULL|    alex|2025-01-03 12:00:00|
|   

## Q4. Filter + GroupBy

In [6]:
df_q4 = df.filter(
    col("region") == "West"
).groupBy(
    "product_category"
).avg(
    "sale_amount"
)

df_q4.show()

+----------------+----------------+
|product_category|avg(sale_amount)|
+----------------+----------------+
|     Electronics|           600.0|
|       Furniture|           700.0|
+----------------+----------------+



## Q5. Fill Null Values

.na.drop()

Removes rows containing null values.

.na.fill()

Replaces null values with specified values.

In [7]:
df_q5 = df.na.fill(
    {"status":"Unknown"}
)

df_q5.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-----+--------+-------+--------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  status|price|store_id|revenue|         email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-----+--------+-------+--------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|    Delhi| 25|     Premium| Unknown|  100|       1|    500|john@gmail.com|    john|2025-01-01 10:00:00|
|    101|      2025-01-01|  West|     Electronics|        500|    Delhi| 25|     Premium| Unknown|  100|       1|    500|john@gmail.com|    john|2025-01-01 10:00:00|
|    102|      2025-01-02|  East|        Clothing|        300|   Mumbai| 19|       Basic|  Active| NULL|       2|    300| sam@gmail.com|     sam|2025-01-02 11:00:00|
|   

## Q6. City Count > 100

In [8]:
df_q6 = df.groupBy(
    "city"
).count().filter(
    col("count") > 100
)

df_q6.show()

+----+-----+
|city|count|
+----+-----+
+----+-----+



## Q7. Immutability
Spark DataFrames are immutable.

Whenever we rename, drop, or modify a column, Spark creates a new DataFrame rather than changing the original one.

Example:

In [9]:
new_df = df.drop("status")

## Q8. Filter Age and Subscription

In [10]:
df_q8 = df.filter(
    (col("age").between(18,30))
    &
    (col("subscription")=="Premium")
)

df_q8.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-----+--------+-------+--------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  status|price|store_id|revenue|         email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-----+--------+-------+--------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|    Delhi| 25|     Premium|    NULL|  100|       1|    500|john@gmail.com|    john|2025-01-01 10:00:00|
|    101|      2025-01-01|  West|     Electronics|        500|    Delhi| 25|     Premium|    NULL|  100|       1|    500|john@gmail.com|    john|2025-01-01 10:00:00|
|    103|      2025-01-03|  West|       Furniture|        700|    Delhi| 28|     Premium|    NULL|  200|       1|    700|          NULL|    alex|2025-01-03 12:00:00|
|   

## Q9. When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?
Handling null values before aggregation prevents:

- Incorrect averages
- Incorrect sums
- Missing values affecting analysis
- Runtime issues

## Q10. Cast Timestamp

In [11]:
df_q10 = df.withColumn(
    "event_time",
    col("raw_timestamp").cast(
        TimestampType()
    )
).drop(
    "raw_timestamp"
)

df_q10.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-----+--------+-------+--------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  status|price|store_id|revenue|         email|username|         event_time|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-----+--------+-------+--------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|    Delhi| 25|     Premium|    NULL|  100|       1|    500|john@gmail.com|    john|2025-01-01 10:00:00|
|    101|      2025-01-01|  West|     Electronics|        500|    Delhi| 25|     Premium|    NULL|  100|       1|    500|john@gmail.com|    john|2025-01-01 10:00:00|
|    102|      2025-01-02|  East|        Clothing|        300|   Mumbai| 19|       Basic|  Active| NULL|       2|    300| sam@gmail.com|     sam|2025-01-02 11:00:00|
|   

## Q11. Shuffle
Shuffle occurs when Spark redistributes data between partitions.

Operations like:
- groupBy()
- join()
- distinct()

cause shuffle.

It is called a wide transformation because data moves across partitions.

## Q12. Remove Invalid Records

In [12]:
df_q12 = df.filter(
    col("email").isNotNull()
).filter(
    trim(col("username")) != ""
)

df_q12.show()

+-------+----------------+------+----------------+-----------+-------+---+------------+------+-----+--------+-------+--------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   city|age|subscription|status|price|store_id|revenue|         email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+-------+---+------------+------+-----+--------+-------+--------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|  Delhi| 25|     Premium|  NULL|  100|       1|    500|john@gmail.com|    john|2025-01-01 10:00:00|
|    101|      2025-01-01|  West|     Electronics|        500|  Delhi| 25|     Premium|  NULL|  100|       1|    500|john@gmail.com|    john|2025-01-01 10:00:00|
|    102|      2025-01-02|  East|        Clothing|        300| Mumbai| 19|       Basic|Active| NULL|       2|    300| sam@gmail.com|     sam|2025-01-02 11:00:00|
|    105|      2025-01-05| S

## Q13. Multiple Aggregations

In [13]:
df_q13 = df.agg(
    min("price").alias("Min Price"),
    max("price").alias("Max Price"),
    mean("price").alias("Average Price")
)

df_q13.show()

+---------+---------+-------------+
|Min Price|Max Price|Average Price|
+---------+---------+-------------+
|      100|      300|        170.0|
+---------+---------+-------------+



## Q14.  In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?
inferSchema=True may incorrectly detect data types when source data contains inconsistent date formats.

This can lead to:
- Null values
- Parsing failures
- Incorrect schema

Explicit schema definition is safer.

## Q15. Final Pipeline


In [14]:
final_df = (
    df
    .dropDuplicates()
    .na.fill({"price":0})
    .groupBy("store_id")
    .agg(
        sum("revenue")
        .alias("total_revenue")
    )
)

final_df.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|       1|         1200|
|       3|          800|
|       2|          700|
+--------+-------------+



## Stop Spark Session

In [15]:
spark.stop()